In [ ]:
import optuna
import pandas as pd
import numpy as np
import lightgbm as lgb
import re
import warnings
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ════════════════════════════════════════════════════════════
# КОНСТАНТЫ
# ════════════════════════════════════════════════════════════
TRAIN_PATH  = 'train_2.csv'
TARGET_DATE = pd.Timestamp('2025-03-01')
VALID_DATE  = pd.Timestamp('2025-02-01')

def competition_score(mape):
    return 100 * ((100 - min(mape, 100)) / 100) ** 2

def calc_mape(y_true, y_pred):
    mask = y_true > 0
    return 100 * np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask]))

def apply_coef(preds):
    return np.where(
        preds < 70_000_000,
        preds * 1.044,
        preds * 1.038
    ) * 0.992

# ════════════════════════════════════════════════════════════
# 1. ЗАГРУЗКА
# ════════════════════════════════════════════════════════════
print("📥 Загрузка...")
df_raw = pd.read_csv(TRAIN_PATH).rename(columns={'РТО': 'target'})
df_raw['timestamp'] = pd.to_datetime(
    df_raw['Год'].astype(str) + '-' +
    df_raw['Месяц'].astype(str).str.zfill(2) + '-01'
)
all_ids = df_raw['new_id'].unique()
print(f"  Магазинов: {len(all_ids)}, строк: {len(df_raw)}")

region_col = None
for c in df_raw.columns:
    if 'егион' in str(c) or 'egion' in str(c).lower():
        region_col = c
        break
if region_col:
    print(f"  Регион: '{region_col}'")

# ════════════════════════════════════════════════════════════
# 2. КЛАСТЕРИЗАЦИЯ
# ════════════════════════════════════════════════════════════
print("\n🔵 Кластеризация...")
df_hist = df_raw[df_raw['timestamp'] < TARGET_DATE].copy()

store_profiles = df_hist.groupby('new_id')['target'].agg(
    mean_rto='mean', std_rto='std', median_rto='median',
    max_rto='max', min_rto='min', count='count',
).fillna(0)
store_profiles['cv']         = store_profiles['std_rto'] / (store_profiles['mean_rto'] + 1)
store_profiles['log_mean']   = np.log1p(store_profiles['mean_rto'])
store_profiles['log_median'] = np.log1p(store_profiles['median_rto'])

zero_counts = df_hist.groupby('new_id')['target'].apply(lambda x: (x == 0).sum())
store_profiles['zero_months'] = store_profiles.index.map(zero_counts).fillna(0)
store_profiles['zero_ratio']  = store_profiles['zero_months'] / store_profiles['count']

last_ts  = sorted(df_hist['timestamp'].unique())
recent_6 = last_ts[-6:]
older_6  = last_ts[-12:-6] if len(last_ts) >= 12 else last_ts[:-6]
mean_recent = df_hist[df_hist['timestamp'].isin(recent_6)].groupby('new_id')['target'].mean()
mean_older  = df_hist[df_hist['timestamp'].isin(older_6)].groupby('new_id')['target'].mean()
store_profiles['trend_6m'] = (
    (mean_recent - mean_older) / (mean_older + 1)
).reindex(store_profiles.index).fillna(0)

march_data = df_hist[df_hist['timestamp'].dt.month == 3].groupby('new_id')['target'].mean()
feb_data   = df_hist[df_hist['timestamp'].dt.month == 2].groupby('new_id')['target'].mean()
store_profiles['mar_vs_feb'] = (
    march_data / (feb_data + 1)
).reindex(store_profiles.index).fillna(1.0)

log_means     = store_profiles['log_mean'].values
q25, q50, q75 = np.percentile(log_means, [25, 50, 75])

def assign_lv1(v):
    if v <= q25:   return 1
    elif v <= q50: return 2
    elif v <= q75: return 3
    else:          return 4

store_profiles['lv1_cluster'] = store_profiles['log_mean'].apply(assign_lv1)

hier_features = ['log_mean', 'cv', 'zero_ratio', 'trend_6m', 'mar_vs_feb']
cluster_final = {}

for lv1 in [1, 2, 3, 4]:
    ids_in = store_profiles[store_profiles['lv1_cluster'] == lv1].index
    n_sub  = 2 if lv1 < 4 else 4
    if len(ids_in) < n_sub * 2:
        for sid in ids_in:
            cluster_final[sid] = f'{lv1}_1'
        continue
    sub_data = store_profiles.loc[ids_in, hier_features].fillna(0).values
    sub_std  = sub_data.std(axis=0)
    sub_std[sub_std == 0] = 1
    sub_norm   = (sub_data - sub_data.mean(axis=0)) / sub_std
    sim_matrix = np.dot(sub_norm, sub_norm.T)
    try:
        Z          = linkage(sim_matrix, method='ward')
        sub_labels = fcluster(Z, t=n_sub, criterion='maxclust')
        for sid, label in zip(ids_in, sub_labels):
            cluster_final[sid] = f'{lv1}_{label}'
    except Exception:
        for sid in ids_in:
            cluster_final[sid] = f'{lv1}_1'

cluster_series  = pd.Series(cluster_final, name='cluster')
cluster_codes   = {c: i for i, c in enumerate(sorted(cluster_series.unique()))}
cluster_numeric = cluster_series.map(cluster_codes)
print(f"  Кластеров: {len(cluster_codes)}")

# ════════════════════════════════════════════════════════════
# 3. FEATURE ENGINEERING (LGB)
# ════════════════════════════════════════════════════════════
def build_features(df):
    df  = df.copy().sort_values(['new_id', 'timestamp']).reset_index(drop=True)
    grp = df.groupby('new_id')['target']

    for lag in [1, 2, 3, 6, 12, 13, 14, 24]:
        df[f'lag_{lag}'] = grp.shift(lag)

    for w in [3, 6, 12]:
        df[f'roll_mean_{w}'] = grp.transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean()
        )
        df[f'roll_std_{w}'] = grp.transform(
            lambda x: x.shift(1).rolling(w, min_periods=2).std()
        )

    for span, name in [(3, '3'), (6, '6'), (12, '12')]:
        df[f'ema_{name}'] = grp.transform(
            lambda x, s=span: x.shift(1).ewm(span=s, adjust=False).mean()
        )

    for alpha, aname in [(0.95, '095'), (0.9, '090'), (0.7, '070'), (0.5, '050')]:
        df[f'ewm_a{aname}'] = grp.transform(
            lambda x, a=alpha: x.shift(1).ewm(alpha=a, adjust=False).mean()
        )

    df['weighted_lag'] = (
        0.50 * grp.shift(1).fillna(0) +
        0.30 * grp.shift(2).fillna(0) +
        0.15 * grp.shift(3).fillna(0) +
        0.05 * grp.shift(6).fillna(0)
    )

    df['yoy_growth']         = df['lag_1']  / (df['lag_12'] + 1e-6)
    df['yoy_ratio_13']       = df['lag_1']  / (df['lag_13'] + 1e-6)
    df['yoy_ratio_2y']       = df['lag_12'] / (df['lag_24'] + 1e-6)
    df['yoy_diff']           = df['lag_1']  - df['lag_13']
    df['mar_feb_ratio_prev'] = df['lag_13'] / (df['lag_14'] + 1e-6)

    df['mom_1m']  = df['lag_1'] / (df['lag_2']  + 1e-6) - 1
    df['mom_3m']  = df['lag_1'] / (df['lag_6']  + 1e-6) - 1
    df['mom_12m'] = df['lag_1'] / (df['lag_13'] + 1e-6) - 1

    store_median         = grp.transform('median')
    store_mean           = grp.transform('mean')
    df['lag1_vs_median'] = df['lag_1'] / (store_median + 1e-6)
    df['lag1_vs_mean']   = df['lag_1'] / (store_mean   + 1e-6)

    df['month']     = df['timestamp'].dt.month
    df['quarter']   = df['timestamp'].dt.quarter
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['is_march']  = (df['month'] == 3).astype(int)
    df['is_q1']     = (df['quarter'] == 1).astype(int)
    df['is_jan']    = (df['month'] == 1).astype(int)

    df['cluster']      = df['new_id'].map(cluster_numeric).fillna(0).astype(int)
    df['zero_ratio_s'] = df['new_id'].map(store_profiles['zero_ratio']).fillna(0)
    df['store_cv']     = df['new_id'].map(store_profiles['cv']).fillna(0)
    df['store_trend']  = df['new_id'].map(store_profiles['trend_6m']).fillna(0)
    df['mar_vs_feb_s'] = df['new_id'].map(store_profiles['mar_vs_feb']).fillna(1)
    df['log_mean_rto'] = df['new_id'].map(store_profiles['log_mean']).fillna(0)

    cl_agg = (
        df.groupby(['cluster', 'timestamp'])['target']
        .agg(cluster_median='median', cluster_mean='mean')
        .reset_index()
    )
    cl_agg['timestamp'] = cl_agg['timestamp'] + pd.DateOffset(months=1)
    df = df.merge(
        cl_agg.rename(columns={
            'cluster_median': 'cluster_median_prev',
            'cluster_mean':   'cluster_mean_prev',
        }),
        on=['cluster', 'timestamp'], how='left'
    )
    df['lag1_vs_cluster'] = df['lag_1'] / (df['cluster_median_prev'] + 1e-6)

    if region_col and region_col in df.columns:
        reg_agg = (
            df.groupby([region_col, 'timestamp'])['target']
            .agg(region_median='median', region_mean='mean')
            .reset_index()
        )
        reg_agg['timestamp'] = reg_agg['timestamp'] + pd.DateOffset(months=1)
        df = df.merge(
            reg_agg.rename(columns={
                'region_median': 'region_median_prev',
                'region_mean':   'region_mean_prev',
            }),
            on=[region_col, 'timestamp'], how='left'
        )
        df['lag1_vs_region'] = df['lag_1'] / (df['region_median_prev'] + 1e-6)

    for c in df.columns:
        if df[c].dtype == 'object' and c not in ['timestamp', 'new_id']:
            df[c] = pd.factorize(df[c].fillna('missing'))[0]

    df = df.rename(
        columns={c: re.sub(r'[^\w]', '_', str(c)) for c in df.columns}
    )
    return df


print("\n🔨 Строим LGB-признаки...")
march_rows = pd.DataFrame({'new_id': all_ids, 'timestamp': TARGET_DATE})
df_full    = pd.concat([df_raw, march_rows], ignore_index=True)\
               .sort_values(['new_id', 'timestamp'])\
               .reset_index(drop=True)

df_feat  = build_features(df_full)
EXCLUDE  = {'target', 'timestamp', 'new_id', 'Год', 'Месяц', 'год', 'месяц'}
features = [c for c in df_feat.columns if c not in EXCLUDE]
print(f"✅ LGB признаков: {len(features)}")

# ── Заполнение NaN ──
train_all   = df_feat[df_feat['target'].notna()].copy()
col_medians = {}
for f in features:
    if f in train_all.columns and pd.api.types.is_numeric_dtype(train_all[f]):
        col_medians[f] = train_all[f].median()

def fill_na(df_in):
    df_out = df_in.copy()
    for f, med in col_medians.items():
        if f in df_out.columns:
            df_out[f] = df_out[f].fillna(med)
    return df_out

train_v = fill_na(df_feat[
    (df_feat['timestamp'] < VALID_DATE) & (df_feat['target'].notna())
])
val_v   = fill_na(df_feat[df_feat['timestamp'] == VALID_DATE])
y_val   = val_v['target'].values

X_train   = train_v[features].values
y_train   = np.log1p(train_v['target'].values)
X_val     = val_v[features].values
y_val_log = np.log1p(val_v['target'].values)

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

# ════════════════════════════════════════════════════════════
# 4. OPTUNA (LGB)
# ════════════════════════════════════════════════════════════
N_TRIALS  = 100
N_ROUNDS  = 2000
ES_ROUNDS = 100

def objective_lgb(trial):
    params = {
        'objective':         'regression_l1',
        'metric':            'mape',
        'verbose':           -1,
        'seed':              42,
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'max_depth':         trial.suggest_int('max_depth', 4, 16),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 300),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
        'max_bin':           trial.suggest_int('max_bin', 128, 512),
        'path_smooth':       trial.suggest_float('path_smooth', 0.0, 1.0),
    }

    dtrain = lgb.Dataset(
        X_train, label=y_train,
        params={'max_bin': params['max_bin']},
        free_raw_data=False,
    )
    dval = lgb.Dataset(
        X_val, label=y_val_log,
        params={'max_bin': params['max_bin']},
        reference=dtrain,
        free_raw_data=False,
    )

    model = lgb.train(
        params, dtrain,
        num_boost_round=N_ROUNDS,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(ES_ROUNDS, verbose=False),
            lgb.log_evaluation(-1),
        ],
    )

    preds = np.expm1(model.predict(X_val)).clip(0)
    mape  = calc_mape(y_val, preds)
    trial.set_user_attr('best_iteration', model.best_iteration)
    return mape


print(f"\n🔬 Optuna LGB: {N_TRIALS} trials...")

study_lgb = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=15),
)

study_lgb.enqueue_trial({
    'num_leaves':        124,
    'max_depth':         12,
    'min_child_samples': 129,
    'learning_rate':     0.02585488501264429,
    'feature_fraction':  0.531610137661962,
    'bagging_fraction':  0.7475039954480004,
    'bagging_freq':      2,
    'reg_alpha':         0.004177468127523882,
    'reg_lambda':        0.006185664836995357,
    'min_gain_to_split': 0.0,
    'max_bin':           255,
    'path_smooth':       0.0,
})

study_lgb.optimize(
    objective_lgb,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    catch=(Exception,),
)

trials_ok = [t for t in study_lgb.trials if t.value is not None]
trials_ok.sort(key=lambda t: t.value)
best_lgb = trials_ok[0]

print(f"\n✅ Лучший LGB trial #{best_lgb.number}")
print(f"   MAPE: {best_lgb.value:.4f}%  |  Балл: {competition_score(best_lgb.value):.2f}")

BEST_PARAMS = {
    'objective':  'regression_l1',
    'metric':     'mape',
    'seed':       42,
    'verbose':    -1,
    **best_lgb.params,
}

# ════════════════════════════════════════════════════════════
# 5. ФИНАЛЬНАЯ LGB-МОДЕЛЬ
# ════════════════════════════════════════════════════════════
print("\n🚀 Финальная LGB-модель...")

train_f  = fill_na(train_all)
march_df = fill_na(df_feat[df_feat['timestamp'] == TARGET_DATE])
ids_lgb  = march_df['new_id'].values

best_iter        = best_lgb.user_attrs.get('best_iteration', 875)
num_rounds_final = int(best_iter * 1.1)

dtrain_f = lgb.Dataset(
    train_f[features].values,
    label=np.log1p(train_f['target'].values),
    params={'max_bin': BEST_PARAMS.get('max_bin', 255)},
    free_raw_data=False,
)

model_f = lgb.train(BEST_PARAMS, dtrain_f, num_boost_round=num_rounds_final)

lgb_preds_raw = np.expm1(model_f.predict(march_df[features].values)).clip(0)
print(f"  LGB: mean={lgb_preds_raw.mean():,.0f}  median={np.median(lgb_preds_raw):,.0f}")

# ════════════════════════════════════════════════════════════
# 6. MLP
# ════════════════════════════════════════════════════════════
def build_nn_features(df):
    df = df.copy().sort_values(['new_id', 'timestamp'])
    grp = df.groupby('new_id')['target']
    df['lag_1']       = grp.shift(1)
    df['lag_12']      = grp.shift(12)
    df['roll_mean_3'] = grp.transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
    df['month']    = df['timestamp'].dt.month
    df['is_march'] = (df['month'] == 3).astype(int)
    for col in ['Регион', 'Населенный пункт']:
        if col in df.columns:
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df.fillna(0)

NN_FEATURES = ['lag_1', 'lag_12', 'roll_mean_3', 'month', 'is_march']

print("\n🚀 MLP финальный прогноз...")
march_grid = pd.DataFrame({'new_id': all_ids, 'timestamp': TARGET_DATE})
df_full_nn = build_nn_features(
    pd.concat([df_raw, march_grid], ignore_index=True)
)

scaler_x = StandardScaler()
scaler_y = StandardScaler()

mask_train_nn = df_full_nn['target'] > 0
X_all_nn = scaler_x.fit_transform(df_full_nn.loc[mask_train_nn, NN_FEATURES])
y_all_nn = scaler_y.fit_transform(
    np.log1p(df_full_nn.loc[mask_train_nn, 'target']).values.reshape(-1, 1)
)
X_march_nn = scaler_x.transform(
    df_full_nn.loc[df_full_nn['timestamp'] == TARGET_DATE, NN_FEATURES]
)

nn_final = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16), max_iter=250, random_state=42
)
nn_final.fit(X_all_nn, y_all_nn.ravel())

mlp_preds = np.expm1(
    scaler_y.inverse_transform(nn_final.predict(X_march_nn).reshape(-1, 1))
).ravel()
ids_mlp = march_grid['new_id'].values
print(f"  MLP: mean={mlp_preds.mean():,.0f}  median={np.median(mlp_preds):,.0f}")

# ════════════════════════════════════════════════════════════
# 7. RIDGE
# ════════════════════════════════════════════════════════════
print("\n🔨 Ridge...")

RIDGE_FEATURES = [
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13',
    'roll_mean_3', 'roll_mean_6', 'roll_mean_12',
    'ema_3', 'ema_6', 'ema_12',
    'month', 'month_sin', 'month_cos',
]

df_ridge = pd.concat([df_raw, march_rows], ignore_index=True)\
             .sort_values(['new_id', 'timestamp'])\
             .reset_index(drop=True)

grp_r = df_ridge.groupby('new_id')['target']

for lag in [1, 2, 3, 6, 12, 13]:
    df_ridge[f'lag_{lag}'] = grp_r.shift(lag)
for w in [3, 6, 12]:
    df_ridge[f'roll_mean_{w}'] = grp_r.transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).mean()
    )
for span, name in [(3, '3'), (6, '6'), (12, '12')]:
    df_ridge[f'ema_{name}'] = grp_r.transform(
        lambda x, s=span: x.shift(1).ewm(span=s, adjust=False).mean()
    )

df_ridge['month']     = df_ridge['timestamp'].dt.month
df_ridge['month_sin'] = np.sin(2 * np.pi * df_ridge['month'] / 12)
df_ridge['month_cos'] = np.cos(2 * np.pi * df_ridge['month'] / 12)

train_r = df_ridge[df_ridge['target'].notna()].copy()
march_r = df_ridge[df_ridge['timestamp'] == TARGET_DATE].copy()

for f in RIDGE_FEATURES:
    med = train_r[f].median()
    train_r[f] = train_r[f].fillna(med)
    march_r[f] = march_r[f].fillna(med)

scaler_r  = StandardScaler()
X_train_r = scaler_r.fit_transform(train_r[RIDGE_FEATURES].values)
X_march_r = scaler_r.transform(march_r[RIDGE_FEATURES].values)
y_train_r = np.log1p(train_r['target'].values)

last_date_r  = train_r['timestamp'].max()
val_mask_r   = (train_r['timestamp'] == last_date_r).values
train_mask_r = ~val_mask_r

def objective_ridge(trial):
    alpha         = trial.suggest_float('alpha', 1e-3, 1e4, log=True)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    solver        = trial.suggest_categorical(
        'solver',
        ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga']
    )
    model = Ridge(alpha=alpha, fit_intercept=fit_intercept,
                  solver=solver, random_state=42)
    model.fit(X_train_r[train_mask_r], y_train_r[train_mask_r])
    preds = model.predict(X_train_r[val_mask_r])
    return np.sqrt(np.mean((y_train_r[val_mask_r] - preds) ** 2))

print("🔍 Optuna Ridge (50 trials)...")
study_ridge = optuna.create_study(direction='minimize')
study_ridge.optimize(objective_ridge, n_trials=50, show_progress_bar=True)

best_ridge_model = Ridge(**study_ridge.best_params, random_state=42)
best_ridge_model.fit(X_train_r, y_train_r)
ridge_preds = np.expm1(best_ridge_model.predict(X_march_r)).clip(0)
ids_ridge   = march_r['new_id'].values
print(f"  Ridge: mean={ridge_preds.mean():,.0f}  median={np.median(ridge_preds):,.0f}")

# ════════════════════════════════════════════════════════════
# 8. БЛЕНДИНГ → ЕДИНСТВЕННЫЙ ФАЙЛ
# ════════════════════════════════════════════════════════════
print("\n🤝 Блендинг lgb×0.44 + mlp×0.51 + ridge×0.05...")

merged = (
    pd.DataFrame({'new_id': ids_lgb, 'lgb': lgb_preds_raw})
    .merge(pd.DataFrame({'new_id': ids_mlp,   'mlp':   mlp_preds}),   on='new_id', how='left')
    .merge(pd.DataFrame({'new_id': ids_ridge,  'ridge': ridge_preds}), on='new_id', how='left')
)

lgb_arr   = merged['lgb'].values
mlp_arr   = merged['mlp'].fillna(merged['lgb']).values
ridge_arr = merged['ridge'].fillna(merged['lgb']).values
ids_all   = merged['new_id'].values

blend = 0.44 * lgb_arr + 0.51 * mlp_arr + 0.05 * ridge_arr
final = apply_coef(blend)

pd.DataFrame({'new_id': ids_all, 'rto': np.round(final, 2)}).to_csv('submission.csv', index=False)

print(f"  mean={final.mean():,.0f}  median={np.median(final):,.0f}")
print(f"\n✅ Готово → submission.csv")